### linear_regression_training()

Traceable sufficient statistics for linear regression.

This cell verifies the `linear_regression_training` function mathematically against its cleartext counterpart, using a dynamically generated input set that covers positive, negative, zero, and array edge cases while respecting the `[-7, 7]` FHE RAM constraints.

In [ ]:
from concrete import fhe
from concrete_fhe_toolkit.ml.trainers import linear_regression_training

def test_linear_regression_training(X_train, y_train, n_samples, n_features):
    import numpy as np
    res = linear_regression_training(X_train, y_train, n_samples, n_features)
    return np.array(res) if isinstance(res, list) else res

compiler = fhe.Compiler(test_linear_regression_training, {'X_train': 'encrypted', 'y_train': 'encrypted', 'n_samples': 'encrypted', 'n_features': 'encrypted'})
inputset = [(2, 2, 2, 2), (0, 0, 0, 0), (1, 1, 1, 1)]
circuit = compiler.compile(inputset)

for inp in inputset:
    try:
        expected = linear_regression_training(inp[0], inp[1], inp[2], inp[3])
        if isinstance(expected, tuple):
            assert tuple(int(x) for x in circuit.encrypt_run_decrypt(*inp)) == expected, f"Failed at {inp}"
        else:
            assert int(circuit.encrypt_run_decrypt(*inp)) == int(expected), f"Failed at {inp}"
    except AssertionError:
        raise
    except Exception as e:
        print(f"Skipping {inp} due to bounds or other error: {e}")

print("linear_regression_training tests passed!")